# Simplified Adult sampling

This notebook shows the smallest MIMIC generation workflow: load Adult, choose `N_ROWS`, call the one-shot `mimic_data(df)` helper, and inspect numeric and categorical distribution diagnostics.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_openml

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src" / "mimic").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import mimic_data
from mimic.diagnostics import categorical_feature_plot, pairwise_feature_plot

RANDOM_STATE = 42
N_ROWS = 200


## Load Adult

`N_ROWS` controls the number of real rows used. The synthetic output has the same number of rows because `mimic_data(df)` defaults to `n_samples=len(df)`.

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True)
raw = adult.frame.copy().replace("?", np.nan)

clean = raw.drop(columns=["fnlwgt", "education-num"]).dropna().reset_index(drop=True)
clean["income"] = clean["class"].astype(str)
clean = clean.drop(columns=["class"])

df = clean.sample(n=N_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

regression_columns = ["age", "hours-per-week", "capital-gain", "capital-loss"]
classification_columns = [c for c in df.columns if c not in regression_columns]

display(df.head())
display(
    pd.DataFrame(
        {
            "quantity": ["rows", "columns", "regression_columns", "classification_columns"],
            "value": [len(df), df.shape[1], len(regression_columns), len(classification_columns)],
        }
    )
)


## One-shot sampling

In [ ]:
synthetic = mimic_data(df)

display(synthetic.head())
display(
    pd.DataFrame(
        {
            "quantity": ["real_rows", "synthetic_rows", "columns_match"],
            "value": [len(df), len(synthetic), list(df.columns) == list(synthetic.columns)],
        }
    )
)


## Numeric diagnostics

In [ ]:
pair_grid = pairwise_feature_plot(
    df,
    synthetic,
    features=regression_columns,
    log1p_features=["capital-gain", "capital-loss"],
    original_label="real",
    generated_label="synthetic",
    max_rows_per_source=300,
    random_state=RANDOM_STATE,
)
pair_grid.fig.suptitle("Pairwise numeric feature statistics: real vs synthetic", y=1.02)


## Categorical diagnostics

In [ ]:
categorical_fig, categorical_axes, categorical_report, categorical_summary = categorical_feature_plot(
    df,
    synthetic,
    features=classification_columns,
    original_label="real",
    generated_label="synthetic",
    top_n=6,
)
categorical_fig.suptitle("Categorical feature proportions: real vs synthetic", y=1.01)

display(
    categorical_summary.sort_values("total_variation_distance", ascending=False)
    .style.format({"total_variation_distance": "{:.3f}"})
)
display(
    categorical_report.sort_values(["feature", "absolute_difference"], ascending=[True, False])
    .style.format(
        {
            "real_proportion": "{:.1%}",
            "synthetic_proportion": "{:.1%}",
            "absolute_difference": "{:.1%}",
        }
    )
)
